# Qwen3.5 4B·9B 오프라인 실행 파일 준비

이 노트북은 `AI_Challenge2_Qwen35_4B_9B_halftrain_analysis.ipynb` 실행에 필요한 모델과 Python wheel을 인터넷이 되는 PC에서 내려받고, 오프라인 PC에서 설치·검증하기 위한 준비용 노트북입니다.

최종 폴더 구조:

```text
downloads/
├─ libs/
└─ models/
   ├─ Qwen3.5-4B/
   └─ Qwen3.5-9B/
```

> **중요:** wheel은 운영체제, CPU 아키텍처, Python 버전에 따라 달라집니다. 가능하면 다운로드 PC와 오프라인 PC에서 같은 OS·아키텍처·Python 버전을 사용하세요. 모델과 wheel을 모두 받을 수 있도록 충분한 저장 공간을 확보하세요.

## 1부. 인터넷이 되는 PC에서 실행

In [ ]:
# Resolve shared code and local assets from the repository root.
import os as _project_os
import sys as _project_sys
from pathlib import Path as _ProjectPath
_project_root = next((p for p in [_ProjectPath.cwd(), *_ProjectPath.cwd().parents]
                      if (p / 'src' / 'ocr_engine.py').is_file()), None)
if _project_root is not None:
    _project_os.chdir(_project_root)
    if str(_project_root / 'src') not in _project_sys.path:
        _project_sys.path.insert(0, str(_project_root / 'src'))
# End repository path setup.

import os
import platform
import sys
from pathlib import Path

DOWNLOAD_DIR = Path("downloads")
LIB_DIR = DOWNLOAD_DIR / "libs"
MODEL_DIR = DOWNLOAD_DIR / "models"

LIB_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Python :", sys.version.split()[0])
print("OS     :", platform.platform())
print("Machine:", platform.machine())
print("libs   :", LIB_DIR.resolve())
print("models :", MODEL_DIR.resolve())

### 1-1. 모델 다운로드 도구 설치

이 셀은 인터넷이 되는 다운로드 PC에서만 실행합니다.

In [ ]:
%pip install -U huggingface_hub

### 1-2. Qwen3.5-4B·9B 다운로드

중간에 중단되더라도 같은 셀을 다시 실행하면 기존 파일을 활용해 이어받습니다.

In [ ]:
from huggingface_hub import snapshot_download

MODELS = {
    "Qwen3.5-4B": "Qwen/Qwen3.5-4B",
    "Qwen3.5-9B": "Qwen/Qwen3.5-9B",
}

for folder_name, repo_id in MODELS.items():
    save_dir = MODEL_DIR / folder_name
    print("=" * 72)
    print("다운로드: ", repo_id)
    print("저장 위치:", save_dir.resolve())

    snapshot_download(
        repo_id=repo_id,
        local_dir=str(save_dir),
    )

    print("완료:", folder_name)

print("\n모든 모델 다운로드 완료")

### 1-3. 모델 파일 검사

`config.json`, tokenizer/processor 설정, safetensors 조각과 index 파일이 있는지 확인합니다.

In [ ]:
def directory_size_gib(path: Path) -> float:
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file()) / 1024**3

required_common = [
    "config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "preprocessor_config.json",
    "model.safetensors.index.json",
]

model_errors = []
for folder_name in MODELS:
    model_path = MODEL_DIR / folder_name
    missing = [name for name in required_common if not (model_path / name).is_file()]
    shards = sorted(model_path.glob("model-*.safetensors"))
    if not shards:
        missing.append("model-*.safetensors")

    status = "OK" if not missing else "ERROR"
    print(f"{folder_name}: {status} / {directory_size_gib(model_path):.2f} GiB / shard {len(shards)}개")
    if missing:
        print("  누락:", missing)
        model_errors.append((folder_name, missing))

assert not model_errors, f"모델 파일 누락: {model_errors}"
print("모델 파일 검사 완료")

### 1-4. 오프라인 설치용 wheel 다운로드

수정된 분석 노트북에 맞춰 Transformers 5.3 이상, PyTorch CUDA 12.8 빌드와 학습·분석 패키지의 의존성을 함께 받습니다. 기존 baseline과 동일하게 PyTorch 2.11 계열을 사용합니다.

> 오프라인 PC의 Python 버전이 다르면 해당 Python 버전으로 이 셀을 다시 실행해야 합니다.

In [ ]:
import subprocess
import sys

packages = [
    # PyTorch / CUDA 12.8
    "torch==2.11.0+cu128",
    "torchvision==0.26.0+cu128",
    "torchaudio==2.11.0+cu128",

    # Qwen3.5 / QLoRA
    "transformers>=5.3.0,<6.0.0",
    "accelerate",
    "peft",
    "bitsandbytes",

    # 학습 / 분석
    "numpy",
    "pandas",
    "pillow",
    "scikit-learn",
    "matplotlib",
    "tqdm",
    "packaging",

    # 모델 파일 및 tokenizer 관련
    "sentencepiece",
    "safetensors",
]

cmd = [
    sys.executable, "-m", "pip", "download",
    "--dest", str(LIB_DIR),
    "--only-binary=:all:",
    "--extra-index-url", "https://download.pytorch.org/whl/cu128",
] + packages

print("라이브러리 wheel 다운로드 시작...")
subprocess.check_call(cmd)
print("완료:", LIB_DIR.resolve())

### 1-5. wheel 결과 검사

핵심 패키지 wheel이 실제로 생성되었는지 확인합니다. 의존성 wheel까지 함께 있으므로 파일 수는 아래 목록보다 많아야 정상입니다.

In [ ]:
wheels = sorted(LIB_DIR.glob("*.whl"))
wheel_names = [p.name.lower().replace("-", "_") for p in wheels]
required_prefixes = [
    "torch_", "torchvision_", "torchaudio_", "transformers_",
    "accelerate_", "peft_", "bitsandbytes_", "numpy_",
    "pandas_", "pillow_", "scikit_learn_", "matplotlib_",
    "tqdm_", "packaging_", "sentencepiece_", "safetensors_",
]
missing_wheels = [
    prefix.rstrip("_")
    for prefix in required_prefixes
    if not any(name.startswith(prefix) for name in wheel_names)
]

libs_size = sum(p.stat().st_size for p in wheels) / 1024**3
print("wheel 개수:", len(wheels))
print(f"libs 크기: {libs_size:.2f} GiB")
print("누락 핵심 wheel:", missing_wheels if missing_wheels else "없음")
assert wheels, "downloads/libs에 wheel이 없습니다."
assert not missing_wheels, f"핵심 wheel 누락: {missing_wheels}"
print("wheel 검사 완료")

### 1-6. 전체 준비 상태 요약

In [ ]:
print("=" * 72)
print("OFFLINE PACKAGE READY")
print("=" * 72)
for folder_name in MODELS:
    path = MODEL_DIR / folder_name
    print(f"{folder_name:12s}: {directory_size_gib(path):6.2f} GiB")
print(f"{'wheel':12s}: {libs_size:6.2f} GiB / {len(wheels)} files")
print("\n이제 downloads 폴더 전체를 오프라인 PC로 복사하세요.")

## 2부. 오프라인 PC에서 실행

`downloads` 폴더와 이 노트북을 같은 작업 폴더에 둔 뒤 아래 셀을 실행합니다. 인터넷을 사용하지 않도록 `--no-index`를 사용합니다.

### 2-1. PyTorch CUDA 12.8 설치

In [ ]:
%pip install --no-index --find-links downloads/libs --upgrade torch==2.11.0+cu128 torchvision==0.26.0+cu128 torchaudio==2.11.0+cu128

### 2-2. Qwen3.5 학습·분석 라이브러리 설치

In [ ]:
%pip install --no-index --find-links downloads/libs --upgrade "transformers>=5.3.0,<6.0.0" accelerate peft bitsandbytes numpy pandas pillow scikit-learn matplotlib tqdm packaging sentencepiece safetensors

### 2-3. 커널 재시작

위 설치가 끝나면 **Jupyter 커널을 반드시 한 번 재시작**한 뒤 아래 검증 셀을 실행하세요. 같은 커널에서 바로 검증하면 설치 전 라이브러리가 메모리에 남아 잘못된 결과가 나올 수 있습니다.

### 2-4. 오프라인 환경·GPU·모델 검증

In [ ]:
import os
from pathlib import Path
from packaging.version import Version

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"

import torch
import transformers
import accelerate
import peft
import bitsandbytes
from transformers import AutoConfig, AutoProcessor

assert Version(transformers.__version__) >= Version("5.3.0"), (
    f"Transformers 5.3 이상이 필요합니다. 현재: {transformers.__version__}"
)
assert torch.cuda.is_available(), (
    "CUDA GPU를 찾지 못했습니다. NVIDIA 드라이버와 CUDA용 PyTorch 설치를 확인하세요."
)
assert "+cu" in torch.__version__, f"CUDA 빌드가 아닌 PyTorch입니다: {torch.__version__}"

model_root = Path("downloads/models")
for folder_name in ["Qwen3.5-4B", "Qwen3.5-9B"]:
    path = model_root / folder_name
    assert (path / "config.json").is_file(), f"모델 없음: {path}"
    config = AutoConfig.from_pretrained(path, local_files_only=True, trust_remote_code=True)
    processor = AutoProcessor.from_pretrained(path, local_files_only=True, trust_remote_code=True)
    print(folder_name, "로컬 로드 OK / model_type =", config.model_type)

print("\nTorch       :", torch.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate  :", accelerate.__version__)
print("PEFT        :", peft.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("GPU         :", torch.cuda.get_device_name(0))
print("VRAM GiB    :", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print("\n오프라인 실행 준비 완료")

## 다음 단계

검증이 끝나면 같은 작업 폴더에서 `AI_Challenge2_Qwen35_4B_9B_halftrain_analysis.ipynb`를 실행합니다. 해당 노트북은 다음 경로를 그대로 사용합니다.

- `downloads/models/Qwen3.5-4B`
- `downloads/models/Qwen3.5-9B`
- `downloads/libs`